In [ ]:
import subprocess
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from packaging.version import Version

# Vá thuộc tính pd.util.version để XGBoost tương thích môi trường Kaggle offline
if not hasattr(pd, "util"):
    class _DummyUtil:
        pass
    pd.util = _DummyUtil()
if not hasattr(pd.util, "version"):
    class _DummyVersion:
        Version = Version
    pd.util.version = _DummyVersion()

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from scipy.stats import rankdata
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBClassifier

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/tnghonglm/read-csv"),
        Path("/kaggle/input/playground-series-s6e11"),
        Path("data"),
    ]
    for p in candidates:
        if (p / "train.csv").exists():
            return p
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        found = next(kaggle_root.rglob("train.csv"), None)
        if found is not None:
            return found.parent
    raise FileNotFoundError("Không tìm thấy train.csv trong /kaggle/input hoặc data/")

DATA_DIR = find_data_dir()

try:
    USE_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    USE_GPU = False

# Trên CPU cá nhân: CatBoost & Neural Net rất chậm nên mặc định chỉ chạy XGB + LGB (~4-5 phút).
# Khi chạy trên Kaggle GPU: tự động bật cả 4 mô hình (XGB + LGB + CAT + NN).
INCLUDE_CATBOOST = USE_GPU
INCLUDE_NN = USE_GPU and HAS_TORCH
print(f"DATA_DIR: {DATA_DIR} | USE_GPU: {USE_GPU} | CAT: {INCLUDE_CATBOOST} | NN: {INCLUDE_NN}")

BOOL_COLS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]

CAT_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Chuyển kiểu boolean sang 0/1, giữ nguyên 24 cột gốc cho GBDT."""
    out = df.copy()
    out[BOOL_COLS] = out[BOOL_COLS].astype(int)
    return out


def add_fold_group_features(
    X_tr: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Tính 7 đặc trưng tổng hợp theo nhóm từ X_tr để tránh rò rỉ dữ liệu (nâng lên 31 cột)."""
    tr, val, te = X_tr.copy(), X_val.copy(), X_test.copy()
    specs = [
        ("merchant_category", "amount_usd", "amount_ratio_cat", "ratio"),
        ("channel", "amount_usd", "amount_ratio_ch", "ratio"),
        ("auth_method", "amount_usd", "amount_ratio_auth", "ratio"),
        ("device_type", "amount_usd", "amount_ratio_dev", "ratio"),
        ("merchant_category", "merchant_risk_score", "risk_diff_cat", "diff"),
        ("merchant_category", "velocity_score", "vel_diff_cat", "diff"),
        ("channel", "velocity_score", "vel_diff_ch", "diff"),
    ]
    for g_col, num_col, new_name, op in specs:
        stats = tr.groupby(g_col, observed=False)[num_col].mean()
        for part in (tr, val, te):
            m_val = part[g_col].map(stats).astype(float)
            part[new_name] = part[num_col] / (m_val + 1.0) if op == "ratio" else part[num_col] - m_val
    return tr, val, te


def add_nn_extra_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo thêm đặc trưng tỷ lệ & chu kỳ thời gian riêng cho mạng Neural Network."""
    out = df.copy()
    out["amount_to_balance_ratio"] = out["amount_usd"] / (out["account_balance_usd"].clip(lower=0) + 1.0)
    out["balance_diff"] = out["account_balance_usd"] - out["amount_usd"]
    out["txn_intensity"] = out["txn_count_last_24h"] / (out["hours_since_last_txn"].clip(lower=0) + 0.1)
    out["travel_speed_kmh"] = out["distance_from_home_km"] / (out["hours_since_last_txn"].clip(lower=0) + 0.1)
    out["risk_flag_count"] = out[BOOL_COLS].sum(axis=1)
    out["cvv_plus_disputes"] = out["cvv_retry_count"] + out["prior_disputes"]
    out["log_amount"] = np.log1p(out["amount_usd"].clip(lower=0))
    out["log_distance"] = np.log1p(out["distance_from_home_km"].clip(lower=0))
    out["log_balance"] = np.log1p(out["account_balance_usd"].clip(lower=0))
    out["hour_sin"] = np.sin(2 * np.pi * out["time_of_day_hour"] / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * out["time_of_day_hour"] / 24.0)
    out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7.0)
    return out

In [ ]:
LR = 0.03 if USE_GPU else 0.05
N_EST = 4000 if USE_GPU else 2000
EARLY_STOP = 100 if USE_GPU else 50

def build_xgb() -> XGBClassifier:
    return XGBClassifier(
        n_estimators=N_EST,
        learning_rate=LR,
        max_depth=5,
        min_child_weight=30,
        subsample=0.8,
        colsample_bytree=0.75,
        reg_alpha=2.0,
        reg_lambda=10.0,
        max_bin=512,
        tree_method="hist",
        device="cuda" if USE_GPU else "cpu",
        enable_categorical=True,
        eval_metric="auc",
        early_stopping_rounds=EARLY_STOP,
        n_jobs=-1,
        random_state=42,
    )


def build_lgb() -> LGBMClassifier:
    return LGBMClassifier(
        n_estimators=N_EST,
        learning_rate=LR,
        num_leaves=31,
        min_child_samples=50,
        max_bin=511,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.75,
        reg_alpha=2.0,
        reg_lambda=10.0,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )


def build_cat() -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=N_EST,
        learning_rate=0.03 if USE_GPU else 0.08,
        depth=7,
        l2_leaf_reg=15.0,
        border_count=254,
        max_ctr_complexity=4 if USE_GPU else 1,
        eval_metric="Logloss",
        task_type="GPU" if USE_GPU else "CPU",
        thread_count=-1,
        random_seed=42,
        verbose=0,
    )


if HAS_TORCH:
    class ResBlock(nn.Module):
        def __init__(self, dim: int, dropout: float = 0.2):
            super().__init__()
            self.block = nn.Sequential(
                nn.LayerNorm(dim),
                nn.SiLU(),
                nn.Linear(dim, dim),
                nn.LayerNorm(dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(dim, dim),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x + self.block(x)

    class TabularResNet(nn.Module):
        def __init__(self, num_features: int, cat_cardinalities: list[int], width: int = 256):
            super().__init__()
            self.embs = nn.ModuleList([
                nn.Embedding(card, min(16, max(4, (card + 1) // 2)))
                for card in cat_cardinalities
            ])
            emb_dim = sum(e.embedding_dim for e in self.embs)
            self.in_proj = nn.Linear(num_features + emb_dim, width)
            self.res1 = ResBlock(width, dropout=0.2)
            self.res2 = ResBlock(width, dropout=0.2)
            self.head = nn.Sequential(
                nn.LayerNorm(width),
                nn.SiLU(),
                nn.Linear(width, 1),
            )

        def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
            emb_outs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
            x = self.in_proj(torch.cat([x_num, *emb_outs], dim=1))
            x = self.res2(self.res1(x))
            return self.head(x).squeeze(1)


def train_predict_nn(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_val: pd.DataFrame, y_val: pd.Series,
    X_test: pd.DataFrame,
    cat_cols: list[str],
    epochs: int = 25,
    batch_size: int = 2048,
) -> tuple[np.ndarray, np.ndarray]:
    """Huấn luyện PyTorch TabularResNet với đặc trưng riêng cho DL + giữ nguyên cột 0/1."""
    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tr_nn = add_nn_extra_features(X_tr)
    X_val_nn = add_nn_extra_features(X_val)
    X_te_nn = add_nn_extra_features(X_test)

    cont_cols = [c for c in X_tr_nn.columns if c not in cat_cols and c not in BOOL_COLS]
    imputer = SimpleImputer(strategy="median")
    scaler = QuantileTransformer(n_quantiles=2000, output_distribution="normal", subsample=150000, random_state=42)
    X_tr_cont = scaler.fit_transform(imputer.fit_transform(X_tr_nn[cont_cols])).astype(np.float32)
    X_val_cont = scaler.transform(imputer.transform(X_val_nn[cont_cols])).astype(np.float32)
    X_te_cont = scaler.transform(imputer.transform(X_te_nn[cont_cols])).astype(np.float32)

    X_tr_num = np.hstack([X_tr_cont, X_tr_nn[BOOL_COLS].to_numpy(dtype=np.float32)])
    X_val_num = np.hstack([X_val_cont, X_val_nn[BOOL_COLS].to_numpy(dtype=np.float32)])
    X_te_num = np.hstack([X_te_cont, X_te_nn[BOOL_COLS].to_numpy(dtype=np.float32)])

    def encode_cats(df_part: pd.DataFrame) -> np.ndarray:
        return np.column_stack([df_part[c].cat.codes.to_numpy(dtype=np.int64) + 1 for c in cat_cols])

    X_tr_cat = encode_cats(X_tr_nn)
    X_val_cat = encode_cats(X_val_nn)
    X_te_cat = encode_cats(X_te_nn)
    cardinalities = [len(X_tr_nn[c].cat.categories) + 1 for c in cat_cols]

    tr_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr_num), torch.from_numpy(X_tr_cat), torch.from_numpy(y_tr.to_numpy(dtype=np.float32))),
        batch_size=batch_size, shuffle=True,
    )

    model = TabularResNet(X_tr_num.shape[1], cardinalities, width=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1.0, None
    val_num_t = torch.from_numpy(X_val_num).to(device)
    val_cat_t = torch.from_numpy(X_val_cat).to(device)

    for _ in range(epochs):
        model.train()
        for xb_num, xb_cat, yb in tr_loader:
            xb_num, xb_cat, yb = xb_num.to(device), xb_cat.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb_num, xb_cat), yb)
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_probs = torch.sigmoid(model(val_num_t, val_cat_t)).cpu().numpy()
        auc = roc_auc_score(y_val, val_probs)
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        val_preds = torch.sigmoid(model(val_num_t, val_cat_t)).cpu().numpy()
        te_preds = []
        for i in range(0, len(X_te_num), 16384):
            te_n = torch.from_numpy(X_te_num[i:i + 16384]).to(device)
            te_c = torch.from_numpy(X_te_cat[i:i + 16384]).to(device)
            te_preds.append(torch.sigmoid(model(te_n, te_c)).cpu().numpy())
    return val_preds, np.concatenate(te_preds)

In [ ]:
# 1. Đọc dữ liệu & đồng bộ category
train = add_features(pd.read_csv(DATA_DIR / "train.csv"))
test = add_features(pd.read_csv(DATA_DIR / "test.csv"))

for col in CAT_COLS:
    cats = pd.CategoricalDtype(categories=sorted(set(train[col].dropna()).union(test[col].dropna())))
    train[col] = train[col].astype(cats)
    test[col] = test[col].astype(cats)

X = train.drop(columns=["id", "is_fraud"])
y = train["is_fraud"].astype(int)
X_test = test.drop(columns=["id"])

# 2. Stratified 5-Fold Cross-Validation cho các mô hình được bật
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = {"xgb": np.zeros(len(X)), "lgb": np.zeros(len(X))}
test_preds = {"xgb": np.zeros(len(X_test)), "lgb": np.zeros(len(X_test))}
if INCLUDE_CATBOOST:
    oof_preds["cat"] = np.zeros(len(X))
    test_preds["cat"] = np.zeros(len(X_test))
if INCLUDE_NN:
    oof_preds["nn"] = np.zeros(len(X))
    test_preds["nn"] = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    t0 = time.time()
    X_tr, X_val, X_te_fold = add_fold_group_features(
        X.iloc[tr_idx], X.iloc[val_idx], X_test
    )
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # --- XGBoost ---
    xgb_model = build_xgb()
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_preds["xgb"][val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_preds["xgb"] += xgb_model.predict_proba(X_te_fold)[:, 1] / n_splits

    # --- LightGBM ---
    lgb_model = build_lgb()
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)],
    )
    oof_preds["lgb"][val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_preds["lgb"] += lgb_model.predict_proba(X_te_fold)[:, 1] / n_splits

    parts = [
        f"Fold {fold}/{n_splits} ({time.time() - t0:.1f}s)",
        f"XGB={roc_auc_score(y_val, oof_preds['xgb'][val_idx]):.5f}",
        f"LGB={roc_auc_score(y_val, oof_preds['lgb'][val_idx]):.5f}",
    ]

    # --- CatBoost ---
    if INCLUDE_CATBOOST:
        cat_model = build_cat()
        cat_model.fit(X_tr, y_tr, cat_features=CAT_COLS, eval_set=(X_val, y_val), early_stopping_rounds=EARLY_STOP)
        oof_preds["cat"][val_idx] = cat_model.predict_proba(X_val)[:, 1]
        test_preds["cat"] += cat_model.predict_proba(X_te_fold)[:, 1] / n_splits
        parts.append(f"CAT={roc_auc_score(y_val, oof_preds['cat'][val_idx]):.5f}")

    # --- PyTorch TabularResNet ---
    if INCLUDE_NN:
        val_nn, te_nn = train_predict_nn(X_tr, y_tr, X_val, y_val, X_te_fold, CAT_COLS)
        oof_preds["nn"][val_idx] = val_nn
        test_preds["nn"] += te_nn / n_splits
        parts.append(f"NN={roc_auc_score(y_val, val_nn):.5f}")

    print(" | ".join(parts))

In [ ]:
# 3. Đánh giá từng mô hình & tối ưu trọng số Ensemble (hỗ trợ 2, 3 hoặc 4 mô hình)
names = list(oof_preds.keys())
for name in names:
    print(f"OOF {name.upper():4s}: {roc_auc_score(y, oof_preds[name]):.5f}")
    np.save(f"oof_{name}.npy", oof_preds[name])
    np.save(f"test_{name}.npy", test_preds[name])

oof_mat = np.column_stack([oof_preds[n] for n in names])
rank_mat = np.column_stack([rankdata(oof_preds[n]) / len(y) for n in names])
test_mat = np.column_stack([test_preds[n] for n in names])
test_rank_mat = np.column_stack([rankdata(test_preds[n]) / len(X_test) for n in names])

# Tạo tập trọng số ứng viên: đều nhau + đơn lẻ + 250 tổ hợp ngẫu nhiên Dirichlet
rng = np.random.default_rng(42)
m = len(names)
w_candidates = [np.full(m, 1.0 / m)]
w_candidates.extend(np.eye(m))
w_candidates.extend(rng.dirichlet(np.ones(m), size=250))
w_candidates = np.array(w_candidates)

# Tìm nhanh top-12 ứng viên trên mẫu phân tầng 150k dòng (nếu >150k), rồi chấm chính xác trên 100% dữ liệu
if len(y) > 150000:
    sss = StratifiedShuffleSplit(n_splits=1, train_size=150000, random_state=42)
    sub_idx, _ = next(sss.split(oof_mat, y))
else:
    sub_idx = np.arange(len(y))
y_sub = y.iloc[sub_idx]

scored = []
for w in w_candidates:
    scored.append((roc_auc_score(y_sub, oof_mat[sub_idx] @ w), "prob", w))
    scored.append((roc_auc_score(y_sub, rank_mat[sub_idx] @ w), "rank", w))
scored.sort(key=lambda x: x[0], reverse=True)

best_auc, best_mode, best_w = -1.0, "prob", np.full(m, 1.0 / m)
for _, mode, w in scored[:12]:
    mat = oof_mat if mode == "prob" else rank_mat
    auc = roc_auc_score(y, mat @ w)
    if auc > best_auc:
        best_auc, best_mode, best_w = auc, mode, w

w_str = ", ".join(f"{n.upper()}={w:.3f}" for n, w in zip(names, best_w))
print(f"\nBest Ensemble ({best_mode}): {w_str}")
print(f"Overall OOF Ensemble ROC-AUC: {best_auc:.5f}")

oof_ens = oof_mat @ best_w
print(classification_report(y, (oof_ens >= 0.5).astype(int)))

test_ens = (test_mat if best_mode == "prob" else test_rank_mat) @ best_w
submission = pd.DataFrame({
    "id": test["id"],
    "is_fraud": test_ens,
})
submission.to_csv("submission.csv", index=False)
print("Đã lưu submission.csv:", submission.shape)